## OpenAI 연동 기본 예제

- `ChatOpenAI`로 OpenAI API에 연결하고, Prompt → LLM → OutputParser를 LCEL로 연결합니다.
- 사전 준비: `.env` 파일에 `OPENAI_API_KEY` 설정, `uv add langchain-openai python-dotenv`

In [ ]:
import os
from dotenv import load_dotenv

#load_dotenv(dotenv_path='.env')
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 키 값은 출력하지 않고 설정 여부만 확인
if not OPENAI_API_KEY:
    raise RuntimeError(".env 파일에 OPENAI_API_KEY 가 없습니다.")
print("OPENAI_API_KEY 설정 확인 완료")

#### OpenAI API로 호출 가능한 모델 목록 확인

- `openai` SDK(`langchain-openai`가 내부적으로 사용)의 `client.models.list()`로 현재 API 키로 사용할 수 있는 모델을 조회합니다.
- 목록 조회는 토큰을 사용하지 않아 비용이 발생하지 않습니다.
- 계정/조직에 따라 목록이 다를 수 있으며, 채팅 모델 외에 임베딩, 이미지, 음성 모델도 함께 나옵니다.

In [ ]:
# openai SDK 불러오기 (langchain-openai가 내부적으로 사용하는 SDK)
from openai import OpenAI

# API 키로 클라이언트 생성 (OPENAI_API_KEY는 load_dotenv()로 .env에서 읽어 둔 값)
client = OpenAI(api_key=OPENAI_API_KEY)

# 호출 가능한 모델 목록 조회 (id 기준 오름차순 정렬)
models = sorted(client.models.list().data, key=lambda m: m.id)
print(f"조회된 모델 수: {len(models)}\n")
for m in models:
    print(f"{m.id:<45} {m.owned_by}")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# prompt
prompt = ChatPromptTemplate.from_messages(
    [ ("system", "당신은 개발자입니다.") , 
     ("user", "{input}") ]
)
print(type(prompt))
print(prompt)

In [ ]:
prompt_text = prompt.format(input="파이썬은 무엇인가요? 자세하게 설명해주세요")
print(type(prompt_text))
print(prompt_text)

In [6]:
from langchain_openai import ChatOpenAI

# 모델명은 상수로 분리
OPENAI_MODEL = "gpt-4o-mini"

# OpenAI API를 사용하는 ChatOpenAI 인스턴스 생성 (base_url 생략 시 OpenAI 기본 엔드포인트 사용)
llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model=OPENAI_MODEL,
    temperature=0.7
)
print(type(llm))
print(llm.model_name)

<class 'langchain_openai.chat_models.base.ChatOpenAI'>
gpt-4o-mini


In [ ]:
try:
    response = llm.invoke(prompt_text)
    print(type(response))
    print("응답:", response.content)
except Exception as e:
    print(f"오류 발생: {e}")

## LCEL(LangChain Expression Language)
### Prompt + LLM 모델 + OutputParser

In [ ]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [ ]:
chain = prompt | llm | output_parser
print(type(chain))
print(chain)

In [ ]:
response = chain.invoke({"input":"LangChain은 무엇인가요?"})

In [ ]:
print(type(response))
print(response)